In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score

In [12]:
df = pd.read_parquet("d:/FIAP/Engenharia de Machine Learning/Fase 3/tech_challenge_3/data/processed/flights_processed.parquet")

In [3]:
pd.set_option('display.max_columns', None) 
df.head()

,AIRLINE,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE_CODE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_DELAY,SCHEDULED_TIME,DISTANCE,SCHEDULED_ARRIVAL,AIRPORT,CITY,STATE,LATITUDE,LONGITUDE,DATE,HOUR,IS_DELAYED,IS_SHORT_DISTANCE,IS_MEDIUM_DISTANCE,IS_LONG_DISTANCE,IS_MORNING,IS_AFTERNOON,IS_NIGHT,IS_HOLIDAY,IS_WINTER,IS_SPRING,IS_SUMMER,IS_FALL
0,AS,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,-11.0,205.0,1448,430,Ted Stevens Anchorage International Airport,Anchorage,AK,61.17432,-149.99619,2015-01-01,0,0,0,1,0,1,0,0,1,1,0,0,0
1,AA,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,-8.0,280.0,2330,750,Los Angeles International Airport,Los Angeles,CA,33.94254,-118.40807,2015-01-01,0,0,0,0,1,1,0,0,1,1,0,0,0
2,US,2015,1,1,4,US,840,N171US,SFO,CLT,20,-2.0,286.0,2296,806,San Francisco International Airport,San Francisco,CA,37.61900,-122.37484,2015-01-01,0,0,0,0,1,1,0,0,1,1,0,0,0
3,AA,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,-5.0,285.0,2342,805,Los Angeles International Airport,Los Angeles,CA,33.94254,-118.40807,2015-01-01,0,0,0,0,1,1,0,0,1,1,0,0,0
4,AS,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,-1.0,235.0,1448,320,Seattle-Tacoma International Airport,Seattle,WA,47.44898,-122.30931,2015-01-01,0,0,0,1,0,1,0,0,1,1,0,0,0


In [13]:
# Suponha que df tenha: TAIL_NUMBER, DATE, SCHEDULED_DEPARTURE, DEPARTURE_DELAY
# Ordenar por aeronave e horário
df = df.sort_values(by=['TAIL_NUMBER', 'DATE', 'SCHEDULED_DEPARTURE'])

# Definir atraso do voo anterior
df['PREVIOUS_DELAY'] = df.groupby('TAIL_NUMBER')['DEPARTURE_DELAY'].shift(1)

# Criar flag de atraso do voo anterior (ex: >= 15 minutos)
df['IS_PREVIOUS_FLIGHT_DELAYED'] = (df['PREVIOUS_DELAY'] >= 15).astype(int)

# Preencher valores NA (primeiro voo de cada aeronave) com 0
df['IS_PREVIOUS_FLIGHT_DELAYED'] = df['IS_PREVIOUS_FLIGHT_DELAYED'].fillna(0)

In [16]:
def create_top_n_feature(
    df, 
    column, 
    target="IS_DELAYED", 
    top_n=10, 
    new_column_name=None,
    drop_original=True
):
    """
    Cria uma coluna categórica "Top N + Other" baseada na proporção de ocorrências
    do target para cada categoria da coluna especificada, compatível com colunas
    categóricas já existentes.

    Parâmetros:
    ----------
    df : pd.DataFrame
        DataFrame original
    column : str
        Nome da coluna que você quer agrupar (ex: 'ORIGIN_AIRPORT')
    target : str, default='IS_DELAYED'
        Coluna alvo usada para calcular o ranking (ex: considerar apenas atrasos)
    top_n : int, default=10
        Número de categorias mais frequentes a manter
    new_column_name : str, default=None
        Nome da nova coluna. Se None, será "{column}_TOP{top_n}"
    drop_original : bool, default=False
        Se True, remove a coluna original após criar a coluna Top N

    Retorna:
    -------
    pd.DataFrame com a nova coluna adicionada
    """
    
    if new_column_name is None:
        new_column_name = f"{column}_TOP{top_n}"

    # Filtrar apenas os casos relevantes do target (ex: voos atrasados)
    filtered = df[df[target] == 1]

    # Contar ocorrências por categoria
    counts = filtered[column].value_counts()

    # Selecionar top N
    top_categories = counts.head(top_n).index

    # Se a coluna for Categorical, adiciona "Other" antes
    if pd.api.types.is_categorical_dtype(df[column]):
        df[column] = df[column].cat.add_categories("Other")

    # Criar coluna nova: mantém top N, resto vira "Other"
    df[new_column_name] = df[column].where(df[column].isin(top_categories), "Other")

    # Remover coluna original, se solicitado
    if drop_original:
        df = df.drop(columns=[column])
    
    return df

In [17]:
# Top 10 aeroportos de origem
df = create_top_n_feature(df, "ORIGIN_AIRPORT", target="IS_DELAYED", top_n=10)
df = create_top_n_feature(df, "DESTINATION_AIRPORT", target="IS_DELAYED", top_n=10)
df = create_top_n_feature(df, "CITY", target="IS_DELAYED", top_n=10)
df = create_top_n_feature(df, "STATE", target="IS_DELAYED", top_n=10)
df = create_top_n_feature(df, "TAIL_NUMBER", target="IS_DELAYED", top_n=10)
df = create_top_n_feature(df, "AIRLINE", target="IS_DELAYED", top_n=10)
df = create_top_n_feature(df, "FLIGHT_NUMBER", target="IS_DELAYED", top_n=10)


C:\Users\alice\AppData\Local\Temp\ipykernel_13096\4131003141.py:47: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(df[column]):
C:\Users\alice\AppData\Local\Temp\ipykernel_13096\4131003141.py:47: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(df[column]):
C:\Users\alice\AppData\Local\Temp\ipykernel_13096\4131003141.py:47: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(df[column]):
C:\Users\alice\AppData\Local\Temp\ipykernel_13096\4131003141.py:47: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead


In [19]:
df.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE_CODE,SCHEDULED_DEPARTURE,DEPARTURE_DELAY,SCHEDULED_TIME,DISTANCE,SCHEDULED_ARRIVAL,...,IS_FALL,PREVIOUS_DELAY,IS_PREVIOUS_FLIGHT_DELAYED,ORIGIN_AIRPORT_TOP10,DESTINATION_AIRPORT_TOP10,CITY_TOP10,STATE_TOP10,TAIL_NUMBER_TOP10,AIRLINE_TOP10,FLIGHT_NUMBER_TOP10
6501,2015,1,1,4,AA,1345,-3.0,85.0,432,1510,...,0,NaN,0,DFW,Other,Dallas-Fort Worth,TX,Other,AA,Other
8425,2015,1,1,4,AA,1550,-4.0,100.0,432,1730,...,0,-3.0,0,Other,DFW,Other,Other,Other,AA,Other
15102,2015,1,2,5,AA,700,-8.0,125.0,731,1005,...,0,-4.0,0,DFW,ATL,Dallas-Fort Worth,TX,Other,AA,Other
18960,2015,1,2,5,AA,1045,-8.0,150.0,731,1215,...,0,-8.0,0,ATL,DFW,Atlanta,GA,Other,AA,Other
21201,2015,1,2,5,AA,1300,8.0,130.0,678,1410,...,0,-8.0,0,DFW,Other,Dallas-Fort Worth,TX,Other,AA,Other


In [20]:
def hhmm_to_minutes(hhmm):
    hhmm = int(hhmm)            # garante que é inteiro
    hours = hhmm // 100          # divide para pegar as horas
    minutes = hhmm % 100         # pega os minutos restantes
    return hours*60 + minutes    # total em minutos

# Exemplo com pandas
df['SCHEDULED_DEPARTURE_MIN'] = df['SCHEDULED_DEPARTURE'].apply(hhmm_to_minutes)

In [25]:
remove = [
    "YEAR", "DAY","LATITUDE", "LONGITUDE", "HOUR", "DATE", "AIRPORT", "AIRLINE_CODE", "SCHEDULED_DEPARTURE"
]

df = df.drop(columns=remove)

In [26]:
categorical_features = [
    "MONTH", "DAY_OF_WEEK", "ORIGIN_AIRPORT_TOP10","DESTINATION_AIRPORT_TOP10",	"CITY_TOP10", "STATE_TOP10"	,"TAIL_NUMBER_TOP10", "AIRLINE_TOP10", "FLIGHT_NUMBER_TOP10"
]

df = pd.get_dummies(df, columns=categorical_features)

MemoryError: Unable to allocate 23.8 GiB for an array with shape (5226569, 4898) and data type bool

In [107]:
from sklearn.preprocessing import StandardScaler
continuous_features = ["DEPARTURE_DELAY", "SCHEDULED_TIME", "DISTANCE", "SCHEDULED_ARRIVAL"]
scaler = StandardScaler()
df[continuous_features] = scaler.fit_transform(df[continuous_features])

In [108]:
df.head()

,DEPARTURE_DELAY,SCHEDULED_TIME,DISTANCE,SCHEDULED_ARRIVAL,IS_DELAYED,IS_SHORT_DISTANCE,IS_MEDIUM_DISTANCE,IS_LONG_DISTANCE,IS_MORNING,IS_AFTERNOON,IS_NIGHT,IS_HOLIDAY,IS_WINTER,IS_SPRING,IS_SUMMER,IS_FALL,SCHEDULED_DEPARTURE_MIN,MONTH_1,MONTH_2,MONTH_3,MONTH_4,MONTH_5,MONTH_6,MONTH_7,MONTH_8,MONTH_9,MONTH_11,MONTH_12,DAY_OF_WEEK_1,DAY_OF_WEEK_2,DAY_OF_WEEK_3,DAY_OF_WEEK_4,DAY_OF_WEEK_5,DAY_OF_WEEK_6,DAY_OF_WEEK_7,ORIGIN_AIRPORT_TOP10_ATL,ORIGIN_AIRPORT_TOP10_DEN,ORIGIN_AIRPORT_TOP10_DFW,ORIGIN_AIRPORT_TOP10_EWR,ORIGIN_AIRPORT_TOP10_IAH,ORIGIN_AIRPORT_TOP10_LAS,ORIGIN_AIRPORT_TOP10_LAX,ORIGIN_AIRPORT_TOP10_ORD,ORIGIN_AIRPORT_TOP10_Other,ORIGIN_AIRPORT_TOP10_PHX,ORIGIN_AIRPORT_TOP10_SFO,DESTINATION_AIRPORT_TOP10_ATL,DESTINATION_AIRPORT_TOP10_DEN,DESTINATION_AIRPORT_TOP10_DFW,DESTINATION_AIRPORT_TOP10_EWR,DESTINATION_AIRPORT_TOP10_IAH,DESTINATION_AIRPORT_TOP10_LAS,DESTINATION_AIRPORT_TOP10_LAX,DESTINATION_AIRPORT_TOP10_ORD,DESTINATION_AIRPORT_TOP10_Other,DESTINATION_AIRPORT_TOP10_PHX,DESTINATION_AIRPORT_TOP10_SFO,CITY_TOP10_Atlanta,CITY_TOP10_Chicago,CITY_TOP10_Dallas-Fort Worth,CITY_TOP10_Denver,CITY_TOP10_Houston,CITY_TOP10_Las Vegas,CITY_TOP10_Los Angeles,CITY_TOP10_New York,CITY_TOP10_Other,CITY_TOP10_Phoenix,CITY_TOP10_San Francisco,STATE_TOP10_AZ,STATE_TOP10_CA,STATE_TOP10_CO,STATE_TOP10_FL,STATE_TOP10_GA,STATE_TOP10_IL,STATE_TOP10_NC,STATE_TOP10_NV,STATE_TOP10_NY,STATE_TOP10_Other,STATE_TOP10_TX,TAIL_NUMBER_TOP10_N361SW,TAIL_NUMBER_TOP10_N364SW,TAIL_NUMBER_TOP10_N365SW,TAIL_NUMBER_TOP10_N394SW,TAIL_NUMBER_TOP10_N637SW,TAIL_NUMBER_TOP10_N638SW,TAIL_NUMBER_TOP10_N640SW,TAIL_NUMBER_TOP10_N641SW,TAIL_NUMBER_TOP10_N646SW,TAIL_NUMBER_TOP10_N647SW,TAIL_NUMBER_TOP10_Other,AIRLINE_TOP10_AA,AIRLINE_TOP10_B6,AIRLINE_TOP10_DL,AIRLINE_TOP10_EV,AIRLINE_TOP10_MQ,AIRLINE_TOP10_NK,AIRLINE_TOP10_OO,AIRLINE_TOP10_Other,AIRLINE_TOP10_UA,AIRLINE_TOP10_US,AIRLINE_TOP10_WN,FLIGHT_NUMBER_TOP10_Other
0,-0.553566,0.835227,1.022515,-2.094129,0,0,1,0,1,0,0,1,1,0,0,0,5,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,True
1,-0.473336,1.830226,2.471147,-1.463845,0,0,0,1,1,0,0,1,1,0,0,0,10,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,True
2,-0.312876,1.909826,2.415304,-1.353545,0,0,0,1,1,0,0,1,1,0,0,0,20,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,True
3,-0.393106,1.896559,2.490857,-1.355515,0,0,0,1,1,0,0,1,1,0,0,0,20,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,Fals

In [121]:
#separa features e target 
X = df.drop(columns=["IS_DELAYED", "DEPARTURE_DELAY"])
y = df["IS_DELAYED"]

In [122]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [123]:
X_train.head()

,SCHEDULED_TIME,DISTANCE,SCHEDULED_ARRIVAL,IS_SHORT_DISTANCE,IS_MEDIUM_DISTANCE,IS_LONG_DISTANCE,IS_MORNING,IS_AFTERNOON,IS_NIGHT,IS_HOLIDAY,IS_WINTER,IS_SPRING,IS_SUMMER,IS_FALL,SCHEDULED_DEPARTURE_MIN,MONTH_1,MONTH_2,MONTH_3,MONTH_4,MONTH_5,MONTH_6,MONTH_7,MONTH_8,MONTH_9,MONTH_11,MONTH_12,DAY_OF_WEEK_1,DAY_OF_WEEK_2,DAY_OF_WEEK_3,DAY_OF_WEEK_4,DAY_OF_WEEK_5,DAY_OF_WEEK_6,DAY_OF_WEEK_7,ORIGIN_AIRPORT_TOP10_ATL,ORIGIN_AIRPORT_TOP10_DEN,ORIGIN_AIRPORT_TOP10_DFW,ORIGIN_AIRPORT_TOP10_EWR,ORIGIN_AIRPORT_TOP10_IAH,ORIGIN_AIRPORT_TOP10_LAS,ORIGIN_AIRPORT_TOP10_LAX,ORIGIN_AIRPORT_TOP10_ORD,ORIGIN_AIRPORT_TOP10_Other,ORIGIN_AIRPORT_TOP10_PHX,ORIGIN_AIRPORT_TOP10_SFO,DESTINATION_AIRPORT_TOP10_ATL,DESTINATION_AIRPORT_TOP10_DEN,DESTINATION_AIRPORT_TOP10_DFW,DESTINATION_AIRPORT_TOP10_EWR,DESTINATION_AIRPORT_TOP10_IAH,DESTINATION_AIRPORT_TOP10_LAS,DESTINATION_AIRPORT_TOP10_LAX,DESTINATION_AIRPORT_TOP10_ORD,DESTINATION_AIRPORT_TOP10_Other,DESTINATION_AIRPORT_TOP10_PHX,DESTINATION_AIRPORT_TOP10_SFO,CITY_TOP10_Atlanta,CITY_TOP10_Chicago,CITY_TOP10_Dallas-Fort Worth,CITY_TOP10_Denver,CITY_TOP10_Houston,CITY_TOP10_Las Vegas,CITY_TOP10_Los Angeles,CITY_TOP10_New York,CITY_TOP10_Other,CITY_TOP10_Phoenix,CITY_TOP10_San Francisco,STATE_TOP10_AZ,STATE_TOP10_CA,STATE_TOP10_CO,STATE_TOP10_FL,STATE_TOP10_GA,STATE_TOP10_IL,STATE_TOP10_NC,STATE_TOP10_NV,STATE_TOP10_NY,STATE_TOP10_Other,STATE_TOP10_TX,TAIL_NUMBER_TOP10_N361SW,TAIL_NUMBER_TOP10_N364SW,TAIL_NUMBER_TOP10_N365SW,TAIL_NUMBER_TOP10_N394SW,TAIL_NUMBER_TOP10_N637SW,TAIL_NUMBER_TOP10_N638SW,TAIL_NUMBER_TOP10_N640SW,TAIL_NUMBER_TOP10_N641SW,TAIL_NUMBER_TOP10_N646SW,TAIL_NUMBER_TOP10_N647SW,TAIL_NUMBER_TOP10_Other,AIRLINE_TOP10_AA,AIRLINE_TOP10_B6,AIRLINE_TOP10_DL,AIRLINE_TOP10_EV,AIRLINE_TOP10_MQ,AIRLINE_TOP10_NK,AIRLINE_TOP10_OO,AIRLINE_TOP10_Other,AIRLINE_TOP10_UA,AIRLINE_TOP10_US,AIRLINE_TOP10_WN,FLIGHT_NUMBER_TOP10_Other
928719,-0.279172,-0.467178,-0.181611,0,1,0,0,1,0,0,1,0,0,0,720,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,True
4357173,3.395691,3.118269,1.654091,0,0,1,0,0,1,0,0,0,0,1,1195,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,True
3499685,-0.637371,-0.587076,-0.855227,1,0,0,1,0,0,0,0,0,1,0,565,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,True
1243093,-0.438372,-0.575579,-1.552479,1,0,0,1,0,0,0,0,1,0,0,376,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,Tr

In [25]:
#from imblearn.under_sampling import RandomUnderSampler
#from collections import Counter
#rus = RandomUnderSampler(random_state=42)
#X_res, y_res = rus.fit_resample(X, y)

#print(Counter(y))
#print(Counter(y_res))

In [126]:
# Ajustando para desbalanceamento
scale = sum(y_train==0) / sum(y_train==1)  # razão de classes
model = xgb.XGBClassifier(
    scale_pos_weight=scale,
    eval_metric="logloss",
    use_label_encoder=False,
    random_state=42
)

# Treinar
model.fit(X_train, y_train)

# Previsão
y_prob = model.predict_proba(X_test)[:,1]

# Ajustando threshold manual ou usando PR curve depois
threshold = 0.533
y_pred = (y_prob >= threshold).astype(int)

# Avaliar
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

c:\Users\alice\AppData\Local\Programs\Python\Python314\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:32:41] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


              precision    recall  f1-score   support

           0       0.88      0.70      0.78   1271054
           1       0.32      0.59      0.41    296917

    accuracy                           0.68   1567971
   macro avg       0.60      0.65      0.60   1567971
weighted avg       0.77      0.68      0.71   1567971

ROC-AUC: 0.7078048808276733


In [125]:
from sklearn.metrics import precision_recall_curve, f1_score
import numpy as np

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
f1_scores = 2 * (precision * recall) / (precision + recall)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
print("Melhor threshold:", best_threshold)

Melhor threshold: 0.5325577


In [ ]:
# Features que têm maior impacto
important_features = [
    "DAY_OF_WEEK",
    "SCHEDULED_ARRIVAL",
    "SCHEDULED_DEPARTURE",
    "FLIGHT_NUMBER",
    "SCHEDULED_TIME",
    "DISTANCE",
    "HOUR",
    "AIRLINE_CODE",
    "DESTINATION_AIRPORT_TOP10",
    "STATE_TOP10",
    "CITY_TOP10",
    "ORIGIN_AIRPORT_TOP10"
]

In [128]:
from sklearn.svm import SVC

# Para SVM precisamos de menos amostras ou subsample, porque é lento
X_sample = X_train_scaled.sample(n=50000, random_state=42)
y_sample = y_train.loc[X_sample.index]

svm = SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42)
svm.fit(X_sample, y_sample)

y_prob = svm.predict_proba(X_test_scaled)[:,1]
y_pred = (y_prob >= threshold).astype(int)

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

In [127]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# Normalizar variáveis contínuas
cont_features = ["DISTANCE", "SCHEDULED_DEPARTURE_MIN", "SCHEDULED_TIME", "SCHEDULED_ARRIVAL"]
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[cont_features] = scaler.fit_transform(X_train[cont_features])
X_test_scaled[cont_features] = scaler.transform(X_test[cont_features])

# Target = tempo de atraso real
y_train_reg = y_train_delay  # por exemplo, DEPARTURE_DELAY
y_test_reg = y_test_delay

# Treinar modelo
reg = LinearRegression()
reg.fit(X_train_scaled, y_train_reg)

# Previsão
y_pred_reg = reg.predict(X_test_scaled)

# Avaliar
mse = mean_squared_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)
print("MSE:", mse)
print("R²:", r2)


MemoryError: Unable to allocate 2.73 GiB for an array with shape (100, 3658598) and data type float64